# Benchmark v1.0 (run after training)

Run this after the training notebook finishes and the model is pushed to HuggingFace.

> Set runtime to **T4 GPU**. Takes ~15 min.

In [ ]:
%%capture
!pip install transformers torch accelerate rouge-score nltk tqdm

In [ ]:
import json
import requests
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

GITHUB_RAW = "https://raw.githubusercontent.com/umarfarookm/transit-foundation-model/main"
HF_REPO_ID = "umarfarookm/UmarTransit-1B"
SYSTEM_PROMPT = "You are UmarTransit-1B, a specialized AI assistant for public transit systems and GTFS (General Transit Feed Specification) data. You provide accurate, detailed answers about transit routes, stops, schedules, transfers, and GTFS concepts."
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1
TOP_P = 0.9

for fname, url in [("benchmark.json", GITHUB_RAW + "/evaluation/benchmark.json"), ("metrics.py", GITHUB_RAW + "/evaluation/metrics.py")]:
    r = requests.get(url)
    with open(fname, "w") as f:
        f.write(r.text)
    print(fname + ": " + str(len(r.text)) + " bytes")

with open("benchmark.json") as f:
    benchmark = json.load(f)
questions = benchmark["questions"]
print("Benchmark: " + str(len(questions)) + " questions")

from metrics import score_response, score_benchmark, print_report, report_to_dict
print("Ready.")

In [ ]:
print("Loading " + HF_REPO_ID + "...")
tok = AutoTokenizer.from_pretrained(HF_REPO_ID)
mdl = AutoModelForCausalLM.from_pretrained(HF_REPO_ID, torch_dtype=torch.bfloat16, device_map="auto")
mdl.eval()
print("GPU: " + str(round(torch.cuda.memory_allocated() / 1024**3, 1)) + " GB")

In [ ]:
def generate(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = mdl.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=True)
    return tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

v1_results = []
for q in tqdm(questions, desc="v1.0"):
    t0 = time.time()
    gen = generate(q["question"])
    elapsed = time.time() - t0
    s = score_response(generated=gen, expected=q["expected_answer"], scoring_criteria=q["scoring_criteria"])
    v1_results.append({
        "id": q["id"], "question": q["question"], "expected_answer": q["expected_answer"],
        "generated": gen, "category": q["category"], "difficulty": q["difficulty"],
        "scoring_criteria": q["scoring_criteria"],
        "rouge_l": round(s.rouge_l, 4), "keyword_match": round(s.keyword_match, 4),
        "criteria_match": round(s.criteria_match, 4), "combined": round(s.combined, 4),
        "time_seconds": round(elapsed, 2),
    })

v1_report = score_benchmark(v1_results, model_id=HF_REPO_ID + " (v1.0)")
print_report(v1_report)
v1_dict = report_to_dict(v1_report)

In [ ]:
v01_dict = {
    "model_id": "umarfarookm/UmarTransit-1B (v0.1)",
    "total_questions": 193,
    "overall": {"rouge_l": 0.3751, "keyword_match": 0.4034, "criteria_match": 0.0721, "combined": 0.2927},
    "categories": {
        "gtfs_terminology": {"combined_avg": 0.3512, "count": 25},
        "gtfs_validation": {"combined_avg": 0.3137, "count": 18},
        "journey_planning": {"combined_avg": 0.2426, "count": 10},
        "route_analysis": {"combined_avg": 0.2901, "count": 75},
        "schedule_reasoning": {"combined_avg": 0.2525, "count": 38},
        "transit_operations": {"combined_avg": 0.3065, "count": 27},
    },
}

base_dict = {
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct (base)",
    "total_questions": 193,
    "overall": {"rouge_l": 0.129, "keyword_match": 0.3678, "criteria_match": 0.0203, "combined": 0.168},
    "categories": {
        "gtfs_terminology": {"combined_avg": 0.342, "count": 25},
        "gtfs_validation": {"combined_avg": 0.2667, "count": 18},
        "journey_planning": {"combined_avg": 0.2967, "count": 10},
        "route_analysis": {"combined_avg": 0.0843, "count": 75},
        "schedule_reasoning": {"combined_avg": 0.1205, "count": 38},
        "transit_operations": {"combined_avg": 0.1927, "count": 27},
    },
}

In [ ]:
print("=" * 85)
print("  THREE-WAY COMPARISON: Base vs v0.1 vs v1.0")
print("=" * 85)
print()
header = "  {:<20} {:>12} {:>12} {:>12} {:>12}".format("Metric", "Base", "v0.1", "v1.0", "v0.1->v1.0")
print(header)
print("  " + "-" * 70)
for metric in ["rouge_l", "keyword_match", "criteria_match", "combined"]:
    b = base_dict["overall"][metric]
    v01 = v01_dict["overall"][metric]
    v1 = v1_dict["overall"][metric]
    delta = v1 - v01
    sign = "+" if delta > 0 else ""
    row = "  {:<20} {:>12.4f} {:>12.4f} {:>12.4f} {:>12}".format(metric, b, v01, v1, sign + "{:.4f}".format(delta))
    print(row)
print()
header2 = "  {:<20} {:>12} {:>12} {:>12} {:>12}".format("Category", "Base", "v0.1", "v1.0", "v0.1->v1.0")
print(header2)
print("  " + "-" * 70)
all_cats = sorted(set(list(base_dict["categories"].keys()) + list(v1_dict["categories"].keys())))
for cat in all_cats:
    b = base_dict["categories"].get(cat, {}).get("combined_avg", 0)
    v01 = v01_dict["categories"].get(cat, {}).get("combined_avg", 0)
    v1 = v1_dict["categories"].get(cat, {}).get("combined_avg", 0)
    delta = v1 - v01
    sign = "+" if delta > 0 else ""
    row = "  {:<20} {:>12.4f} {:>12.4f} {:>12.4f} {:>12}".format(cat, b, v01, v1, sign + "{:.4f}".format(delta))
    print(row)
print("=" * 85)
overall_delta = v1_dict["overall"]["combined"] - v01_dict["overall"]["combined"]
print()
print("  v0.1 -> v1.0 overall: " + ("+" if overall_delta > 0 else "") + "{:.4f}".format(overall_delta))
print("  v1.0 vs base: +" + "{:.4f}".format(v1_dict["overall"]["combined"] - base_dict["overall"]["combined"]))

In [ ]:
from datetime import datetime, timezone

v1_output = {"evaluated_at": datetime.now(timezone.utc).isoformat(), "report": v1_dict, "results": v1_results}
with open("umartransit_v1_results.json", "w") as f:
    json.dump(v1_output, f, indent=2, ensure_ascii=False)

comparison = {
    "evaluated_at": datetime.now(timezone.utc).isoformat(),
    "questions_evaluated": len(questions),
    "base_model": base_dict,
    "umartransit_v01": v01_dict,
    "umartransit_v1": v1_dict,
}
with open("v1_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print("Saved:")
print("  1. umartransit_v1_results.json")
print("  2. v1_comparison.json")
print()
print("Download both from the Colab file browser (left panel).")